# 10 - Fault diagnosis and sensor-based state estimation

**Week 11; suggested duration: two teaching periods**

## Learning objectives

- Distinguish process, sensor and actuator faults
- Construct a residual using a reference signal
- Implement threshold and EWMA detectors
- Interpret sensitivity, specificity and detection delay
- Prepare diagnostic evidence for Bayesian updating

**PUNDIT connection.** This unit supports the educational exploitation of PUNDIT (`2020-1.2.3-EUREKA-2022-00021`). The teaching material alone does not establish fulfillment of the original contractual technical deliverables. All TK-101 numerical inputs are synthetic unless explicitly stated otherwise.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Repository root:", ROOT)


## 1. Detection, isolation and identification

Three diagnostic questions: **is there a fault, where is it, and what is its type or magnitude?**

Examples: abnormal inflow as a process fault; LT-101 bias, drift or sticking as a sensor fault; failure of XV-101 to close as an actuator fault; and an inadequate reference model as model mismatch.


## 2. Synthetic TK-101 data

`sensor_data.csv` is synthetic teaching data. `true_level_pct` is an illustrative reference: a real plant's true state is not normally observed directly without uncertainty.


In [ ]:
path=ROOT/'data'/'pundit_case'/'sensor_data.csv'; df=pd.read_csv(path); df.head()


In [ ]:
plt.plot(df.time_min,df.true_level_pct,label='reference'); plt.plot(df.time_min,df.lt101_measured_pct,label='LT-101',alpha=.8); plt.xlabel('time [min]'); plt.ylabel('level [%]'); plt.legend(); plt.grid(alpha=.3); plt.show()


## 3. Residual

Use $r(t)=y_{measured}(t)-y_{expected}(t)$. A growing negative residual may suggest failed-low behavior or bias, but the residual alone does not prove its cause.


In [ ]:
df['residual']=df.lt101_measured_pct-df.true_level_pct
plt.plot(df.time_min,df.residual); plt.axhline(0,ls='--'); plt.xlabel('time [min]'); plt.ylabel('residual'); plt.grid(alpha=.3); plt.show()


## 4. Threshold detector

The first detector signals fault evidence when $r(t)\le -3$. This threshold is a synthetic teaching parameter.


In [ ]:
threshold=-3.; df['detected']=df.residual<=threshold
pd.crosstab(df.fault_active.astype(bool),df.detected,rownames=['fault'],colnames=['detected'])


## 5. Diagnostic metrics

$TPR=TP/(TP+FN)$ is sensitivity/recall; $TNR=TN/(TN+FP)$ is specificity. False-negative and false-positive consequences need not be equal.


In [ ]:
def metrics(y,p):
 y=np.asarray(y,bool); p=np.asarray(p,bool); tp=np.sum(y&p); tn=np.sum(~y&~p); fp=np.sum(~y&p); fn=np.sum(y&~p)
 return {'TP':tp,'TN':tn,'FP':fp,'FN':fn,'sensitivity':tp/(tp+fn) if tp+fn else np.nan,'specificity':tn/(tn+fp) if tn+fp else np.nan}
metrics(df.fault_active,df.detected)


## 6. Threshold trade-off

Sweep the threshold. A more restrictive threshold can reduce false positives while increasing missed detections.


In [ ]:
rows=[]
for th in np.linspace(-.5,-8,50): rows.append({'threshold':th,**metrics(df.fault_active,df.residual<=th)})
perf=pd.DataFrame(rows); plt.plot(perf.threshold,perf.sensitivity,label='sensitivity'); plt.plot(perf.threshold,perf.specificity,label='specificity'); plt.legend(); plt.grid(alpha=.3); plt.show()


## 7. EWMA

$z_t=\alpha r_t+(1-\alpha)z_{t-1}$. A smaller $\alpha$ smooths the signal more but can delay detection.


In [ ]:
def ewma(x,alpha=.12):
 x=np.asarray(x,float); z=np.empty_like(x); z[0]=x[0]
 for i in range(1,len(x)): z[i]=alpha*x[i]+(1-alpha)*z[i-1]
 return z
df['ewma']=ewma(df.residual); plt.plot(df.time_min,df.residual,alpha=.35); plt.plot(df.time_min,df.ewma); plt.axhline(-3,ls='--'); plt.grid(alpha=.3); plt.show()


## 8. Detection delay

In addition to sensitivity, assess the time between fault onset and available evidence.


In [ ]:
fault_start=df.loc[df.fault_active.eq(1),'time_min'].min(); first=df.loc[df.ewma.le(-3),'time_min'].min(); print('fault start',fault_start,'first detection',first,'delay',first-fault_start)


## 9. Diagnostic evidence -> Bayesian likelihood

Detector sensitivity and false-positive rate can estimate $P(E|F)$ and $P(E|\neg F)$. Calibrating and evaluating on the same data is optimistic; use independent validation or a suitable train/validation/test split for research claims.

Zero false positives in this small synthetic sample do not establish a population false-positive rate of zero.


In [ ]:
m=metrics(df.fault_active,df.detected); sens=m['sensitivity']; fp=1-m['specificity']; prior=.02
post=sens*prior/(sens*prior+fp*(1-prior)) if fp>0 else 1.
print({'sensitivity':sens,'false_positive':fp,'posterior_if_alarm':post})


In [ ]:
from safetycourse.diagnosis import residual,threshold_detector,ewma as lib_ewma,binary_metrics
r=residual(df.lt101_measured_pct,df.true_level_pct); pred=threshold_detector(r,-3.,direction='low'); print(binary_metrics(df.fault_active,pred)); assert np.allclose(lib_ewma(r,.12),df.ewma)


## Try it!

1. Compare detection delays for alpha=0.05, 0.2 and 0.5.
2. Select a threshold giving at least 0.90 sensitivity and report specificity.
3. Increase the fault prior to 0.10 and calculate the posterior.
4. Explain what could replace the true_level reference in a real process.


## Summary

Diagnosis creates evidence from sensor data. Next, translate this evidence into **time-varying risk and maintenance decision inputs**, with explicit assumptions.
